In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
import datetime as _dt

In [0]:
try:
    arrival_date = dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date = _dt.date.today().strftime("%Y-%m-%d")

try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "travel_bookings"
try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "default"

In [0]:
src = spark.table(f"{catalog}.bronze.customers_inc").where(F.col("business_date") == F.to_date(F.lit(arrival_date)))

dim_full_name = f"{catalog}.{schema}.customer_dim"


In [0]:
spark.sql(f"create schema if not exists {catalog}.{schema}")
spark.sql(f"""create table if not exists {dim_full_name} (
    customer_sk bigint generated always as identity,
    customer_id integer,
    customer_name string,
    customer_address string,
    email string,
    valid_from date,
    valid_to date,
    is_current boolean
)using delta""")


if spark.catalog.tableExists(dim_full_name):
    dim = DeltaTable.forName(spark,dim_full_name)
    (dim.alias("t").merge(src.alias("s"),"t.customer_id = s.customer_id AND t.is_current = true")
    .whenMatchedUpdate(
        condition="t.customer_name <> s.customer_name or t.customer_address <> s.customer_address or t.email <> s.email",
        set ={"valid_to":"s.valid_from","is_current":"false"}
    )
    .whenNotMatchedInsert(values = {
        "customer_id":"s.customer_id",
        "customer_name":"s.customer_name",
        "customer_address":"s.customer_address",
        "email":"s.email",
        "valid_from":"s.valid_from",
        "valid_to":"s.valid_to",
        "is_current":"true"
    })
    .execute())

    changed = spark.sql(f"""
                        select s.customer_id,s.customer_name,s.customer_address,s.email,s.valid_from,s.valid_to, true as is_current
                        from {catalog}.bronze.customers_inc s
                        left join {dim_full_name} t
                        on t.customer_id = s.customer_id and t.valid_from = s.valid_from and t.is_current = true
                        where s.business_date = DATE('{arrival_date}') and t.customer_id  is null
                        """)
    if changed.count() > 0:
        changed.write.mode("append").format("delta").saveAsTable(dim_full_name)

else:
    init = src.select("customer_id","customer_name","customer_address","email","valid_from","valid_to","is_current")
    init.write.mode("append").format("delta").saveAsTable(dim_full_name)
    print("SCD Type 2 Merge Completed")
